# Notebook 17: Poster Composition — "India Breathes"

**The poster.** Four visual panels + bold key numbers, McKinsey-style.

```
INDIA BREATHES
One Grid · 1,777 TWh · 366 Days · 2024

  97%            +63%           250 GW
  shortage       demand in      peak demand
  collapsed      a decade       record

  ┄┄ MELTING MOUNTAIN (shortage ridges 2013→2025) ┄┄

  0→327         73%            1.8×
  days>200K     still coal     monsoon cleaner

  ┄┄ THRESHOLD BREACH (peak demand heatmap) ┄┄

  ┄┄ THE OVERTAKE (solar+wind vs gas bars) ┄┄

  ┄┄ REGIONAL HEARTBEATS (5 regions × fuel) ┄┄

  2.3×          1.35 Bt        9.7×
  wind+solar    CO₂ from       the gap
  in 5 years    power          in 2025
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import warnings
warnings.filterwarnings('ignore')

bg_color = '#FAFAF5'
month_starts = [0, 31, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335]
month_labels = ['J', 'F', 'M', 'A', 'M', 'J', 'J', 'A', 'S', 'O', 'N', 'D']

# ── Load data ──
posoco = pd.read_csv('../data/raw/POSOCO_data.csv')
posoco['date'] = pd.to_datetime(posoco['yyyymmdd'], format='%Y%m%d')
posoco['year'] = posoco['date'].dt.year
posoco['doy'] = posoco['date'].dt.dayofyear

cea = pd.read_csv('../data/processed/india_all_years.csv', parse_dates=['date'])
cea['year'] = cea['date'].dt.year
cea['doy'] = cea['date'].dt.dayofyear

d24 = pd.read_csv('../data/processed/india_2024_derived.csv', parse_dates=['date'])

print("Data loaded. Building poster...")

In [ ]:
# ═══════════════════════════════════════════════════════════
# POSTER COMPOSITION — "India Breathes" v2
# ═══════════════════════════════════════════════════════════

# Palettes
fuel_palette = {
    'Coal': '#3D2B1F', 'Gas': '#4A90D9', 'Hydro': '#1B4F72',
    'Nuclear': '#2EC4B6', 'RES': '#F5B041',
}
fuel_types = ['Coal', 'Gas', 'Hydro', 'Nuclear', 'RES']

shortage_cmap = LinearSegmentedColormap.from_list('shortage_yr',
    ['#922B21', '#C0392B', '#F5B041', '#2EC4B6', '#1B8A7A'], N=13)

# Threshold colormap
bounds = [100000, 150000, 175000, 200000, 225000, 260000]
colors_bands = ['#FEF3E2', '#FDEBD0', '#F5B041', '#E74C3C', '#7B241C']
band_cmap = LinearSegmentedColormap.from_list('threshold', colors_bands, N=256)
band_norm = mcolors.BoundaryNorm(bounds, band_cmap.N)

# ── Helper: draw a key number callout ──
def draw_callout(ax, big_num, label, sub_label='', color='#333'):
    ax.set_facecolor(bg_color)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    ax.text(0.5, 0.62, big_num, ha='center', va='center',
            fontsize=36, fontweight='bold', color=color,
            fontfamily='sans-serif')
    ax.text(0.5, 0.28, label, ha='center', va='center',
            fontsize=11, color='#555', fontweight='bold')
    if sub_label:
        ax.text(0.5, 0.10, sub_label, ha='center', va='center',
                fontsize=8, color='#999', style='italic')

# ── Precompute solar+wind vs gas data ──
sw_gas_years = list(range(2020, 2026))
solar_annual, wind_annual, gas_annual = [], [], []
for yr in sw_gas_years:
    sub = cea[cea['year'] == yr]
    solar_annual.append(sub['solar'].sum() / 1000)   # TWh
    wind_annual.append(sub['wind'].sum() / 1000)
    gas_annual.append(sub['gas'].sum() / 1000)

# ═══════════════════════════════════════════════════════════
# BUILD THE POSTER
# ═══════════════════════════════════════════════════════════

fig = plt.figure(figsize=(24, 40), facecolor=bg_color, dpi=150)

# Master grid: title, KN1, melting mountain, KN2, threshold, solar vs gas, regional, KN3, footer
gs = gridspec.GridSpec(9, 1,
    height_ratios=[1.0, 0.9, 3.2, 0.9, 2.2, 1.8, 2.8, 0.9, 0.35],
    hspace=0.06, figure=fig)

# ────────────────────────────────
# ROW 0: Title
# ────────────────────────────────
ax_title = fig.add_subplot(gs[0])
ax_title.set_facecolor(bg_color)
ax_title.axis('off')
ax_title.text(0.5, 0.65, 'INDIA BREATHES', ha='center', va='center',
              fontsize=48, fontweight='bold', color='#1B4F72', fontfamily='sans-serif')
ax_title.text(0.5, 0.20, 'One Grid  ·  1,777 TWh  ·  366 Days  ·  2024',
              ha='center', va='center', fontsize=16, color='#999', fontfamily='sans-serif')

# ────────────────────────────────
# ROW 1: Key numbers (top)
# ────────────────────────────────
gs_kn1 = gs[1].subgridspec(1, 3, wspace=0.15)

ax_k1 = fig.add_subplot(gs_kn1[0, 0])
draw_callout(ax_k1, '97%', 'shortage collapsed', '5,806 → 172 MW peak (2013–2025)', color='#2EC4B6')

ax_k2 = fig.add_subplot(gs_kn1[0, 1])
draw_callout(ax_k2, '+63%', 'demand in a decade', '45.6M → 74.5M MU (2014–2025)', color='#E74C3C')

ax_k3 = fig.add_subplot(gs_kn1[0, 2])
draw_callout(ax_k3, '250 GW', 'peak demand record', 'set June 2024', color='#1B4F72')

# ────────────────────────────────
# ROW 2: Melting Mountain
# ────────────────────────────────
ax_melt = fig.add_subplot(gs[2])
ax_melt.set_facecolor(bg_color)

shortage_years = list(range(2013, 2026))
all_shortage = posoco[posoco['year'].isin(shortage_years)]['India: PeakShortage'].dropna()
gmax_s = all_shortage.max()

for idx, yr in enumerate(shortage_years):
    sub = posoco[posoco['year'] == yr][['doy', 'India: PeakShortage']].dropna().sort_values('doy')
    if len(sub) == 0:
        continue
    x = sub['doy'].values
    y = sub['India: PeakShortage'].values / gmax_s
    y_s = pd.Series(y).rolling(7, center=True, min_periods=1).mean().values
    y_off = idx * 0.75
    color = shortage_cmap(idx / (len(shortage_years) - 1))
    ax_melt.fill_between(x, y_off, y_off + y_s * 0.65, color=color, alpha=0.8, linewidth=0)
    ax_melt.plot(x, y_off + y_s * 0.65, color=color, linewidth=0.5, alpha=0.9)
    ax_melt.text(-8, y_off + 0.1, str(yr), fontsize=9, fontweight='bold', ha='right', va='center', color='#333')
    avg = sub['India: PeakShortage'].mean()
    ax_melt.text(372, y_off + 0.1, f'{avg:,.0f}', fontsize=8, ha='left', va='center', color='#999')

ax_melt.text(372, len(shortage_years) * 0.75 + 0.15, 'Avg MW', fontsize=8, ha='left', color='#666', fontweight='bold')
ax_melt.set_xticks(month_starts)
ax_melt.set_xticklabels(month_labels, fontsize=8, color='#999')
ax_melt.set_xlim(-5, 400)
ax_melt.set_ylim(-0.15, len(shortage_years) * 0.75 + 0.3)
ax_melt.set_yticks([])
ax_melt.spines[:].set_visible(False)
ax_melt.set_title('The Mountain Melts — Peak Electricity Shortage', fontsize=13,
                   fontweight='bold', color='#333', pad=10, loc='left')

# ────────────────────────────────
# ROW 3: Key numbers (middle)
# ────────────────────────────────
gs_kn2 = gs[3].subgridspec(1, 3, wspace=0.15)

ax_k4 = fig.add_subplot(gs_kn2[0, 0])
draw_callout(ax_k4, '0 → 327', 'days above 200K MW', '2017 → 2024', color='#7B241C')

ax_k5 = fig.add_subplot(gs_kn2[0, 1])
draw_callout(ax_k5, '73%', 'still coal', 'coal share of generation (2024)', color='#3D2B1F')

ax_k6 = fig.add_subplot(gs_kn2[0, 2])
draw_callout(ax_k6, '1.8×', 'monsoon is cleaner', 'Aug 31% clean vs Jan 17%', color='#2EC4B6')

# ────────────────────────────────
# ROW 4: Threshold Breach Heatmap
# ────────────────────────────────
ax_thresh = fig.add_subplot(gs[4])
ax_thresh.set_facecolor('#EEE')

threshold_years = list(range(2017, 2026))
max_dm = np.full((len(threshold_years), 366), np.nan)
for i, yr in enumerate(threshold_years):
    sub = posoco[posoco['year'] == yr][['doy', 'India: MaximumDemand']].dropna()
    for _, row in sub.iterrows():
        d = int(row['doy']) - 1
        if d < 366:
            max_dm[i, d] = row['India: MaximumDemand']

im = ax_thresh.imshow(max_dm, aspect='auto', cmap=band_cmap, norm=band_norm, interpolation='nearest')
ax_thresh.set_yticks(range(len(threshold_years)))
ax_thresh.set_yticklabels(threshold_years, fontsize=9, fontweight='bold')
ax_thresh.set_xticks(month_starts)
ax_thresh.set_xticklabels(month_labels, fontsize=8, color='#999')
ax_thresh.spines[:].set_visible(False)
ax_thresh.set_title('The Hot Zone Expands — Peak Demand', fontsize=13,
                     fontweight='bold', color='#333', pad=10, loc='left')

cbar = plt.colorbar(im, ax=ax_thresh, shrink=0.3, pad=0.01, aspect=15)
cbar.set_ticks(bounds)
cbar.set_ticklabels([f'{b//1000}K' for b in bounds])
cbar.ax.tick_params(labelsize=7)

# ────────────────────────────────
# ROW 5: Solar + Wind vs Gas
# ────────────────────────────────
ax_sw = fig.add_subplot(gs[5])
ax_sw.set_facecolor(bg_color)

x_pos = np.arange(len(sw_gas_years))
bar_w = 0.35

# Stacked solar + wind
bars_wind = ax_sw.bar(x_pos - bar_w/2, wind_annual, bar_w, color='#85C1E9', alpha=0.9, label='Wind')
bars_solar = ax_sw.bar(x_pos - bar_w/2, solar_annual, bar_w, bottom=wind_annual,
                        color='#F5B041', alpha=0.9, label='Solar')

# Gas
bars_gas = ax_sw.bar(x_pos + bar_w/2, gas_annual, bar_w, color='#4A90D9', alpha=0.9, label='Gas')

# Ratio annotations on top of solar+wind bars
for i, (s, w, g) in enumerate(zip(solar_annual, wind_annual, gas_annual)):
    total_sw = s + w
    ratio = total_sw / g
    ax_sw.text(x_pos[i] - bar_w/2, total_sw + 3, f'{ratio:.0f}×',
               ha='center', va='bottom', fontsize=12, fontweight='bold', color='#333')

ax_sw.set_xticks(x_pos)
ax_sw.set_xticklabels(sw_gas_years, fontsize=11, fontweight='bold')
ax_sw.set_ylabel('TWh', fontsize=10, fontweight='bold')
ax_sw.spines['top'].set_visible(False)
ax_sw.spines['right'].set_visible(False)
ax_sw.legend(fontsize=9, loc='upper left', frameon=False)
ax_sw.set_title('The Overtake — Solar + Wind vs Gas', fontsize=13,
                 fontweight='bold', color='#333', pad=10, loc='left')

# ────────────────────────────────
# ROW 6: Regional Heartbeats
# ────────────────────────────────
gs_reg = gs[6].subgridspec(5, 1, hspace=0.05)

regions = ['NR', 'WR', 'SR', 'ER', 'NER']
region_names = {'NR': 'Northern', 'WR': 'Western', 'SR': 'Southern', 'ER': 'Eastern', 'NER': 'North-East'}
p24 = posoco[posoco['year'] == 2024].copy().sort_values('doy')

gmax_r = max(p24[f'{r}: Total'].max() for r in regions if f'{r}: Total' in p24.columns)

for idx, region in enumerate(regions):
    ax = fig.add_subplot(gs_reg[idx])
    ax.set_facecolor(bg_color)
    x = p24['doy'].values
    bottom = np.zeros(len(p24))
    for fuel in fuel_types:
        col = f'{region}: {fuel}'
        if col in p24.columns:
            vals = pd.Series(p24[col].fillna(0).values).rolling(7, center=True, min_periods=1).mean().values
            ax.fill_between(x, bottom, bottom + vals, color=fuel_palette[fuel], alpha=0.85, linewidth=0)
            bottom += vals
    ax.set_ylim(0, gmax_r * 1.05)
    ax.set_xlim(1, 366)
    ax.set_yticks([])
    ax.spines[:].set_visible(False)
    ax.set_ylabel(region, fontsize=10, fontweight='bold', rotation=0, labelpad=25, va='center', color='#333')
    for ms in month_starts:
        ax.axvline(ms, color='#DDD', linewidth=0.2, zorder=0)
    if idx == 0:
        ax.set_title('Five Grids, Five Personalities — Regional Generation by Source (2024)',
                      fontsize=13, fontweight='bold', color='#333', pad=10, loc='left')
    if idx < 4:
        ax.set_xticks([])
    else:
        ax.set_xticks(month_starts)
        ax.set_xticklabels(month_labels, fontsize=8, color='#999')

# ────────────────────────────────
# ROW 7: Key numbers (bottom)
# ────────────────────────────────
gs_kn3 = gs[7].subgridspec(1, 3, wspace=0.15)

ax_k7 = fig.add_subplot(gs_kn3[0, 0])
draw_callout(ax_k7, '2.3×', 'wind+solar in 5 years', '110 → 250 TWh (2020–2025)', color='#F5B041')

ax_k8 = fig.add_subplot(gs_kn3[0, 1])
draw_callout(ax_k8, '1.35 Bt', 'CO₂ from power', 'estimated 2024 emissions', color='#922B21')

ax_k9 = fig.add_subplot(gs_kn3[0, 2])
draw_callout(ax_k9, '9.7×', 'the gap in 2025', 'solar+wind now 10× gas', color='#F5B041')

# ────────────────────────────────
# ROW 8: Footer
# ────────────────────────────────
ax_footer = fig.add_subplot(gs[8])
ax_footer.set_facecolor(bg_color)
ax_footer.axis('off')

for i, fuel in enumerate(fuel_types):
    ax_footer.add_patch(plt.Rectangle((0.05 + i * 0.12, 0.55), 0.03, 0.3,
                        color=fuel_palette[fuel], alpha=0.85, transform=ax_footer.transAxes))
    ax_footer.text(0.09 + i * 0.12, 0.7, fuel, fontsize=8, va='center', color='#666',
                   transform=ax_footer.transAxes)

ax_footer.text(0.5, 0.15, 'Data: CEA Daily Generation Report & POSOCO (via Robbie Andrew)  |  "One Sensor, One Year" — Edition 1',
               ha='center', va='center', fontsize=8, color='#BBB', transform=ax_footer.transAxes)

plt.savefig('../art/output/poster_india_breathes_v2.png', dpi=150, bbox_inches='tight', facecolor=bg_color)
plt.show()
print("Poster v2 saved to art/output/poster_india_breathes_v2.png")